In [1]:
!python --version


Python 3.13.15


In [2]:
import requests

url = "http://books.toscrape.com/"
response = requests.get(url)

print(response.status_code)

200


In [3]:
print(response.text[:500])

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


In [4]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title)

<title>
    All products | Books to Scrape - Sandbox
</title>


In [5]:
print(soup.title.text)



    All products | Books to Scrape - Sandbox



In [6]:
print(soup.title)
print(soup.title.text)

<title>
    All products | Books to Scrape - Sandbox
</title>

    All products | Books to Scrape - Sandbox



In [7]:
books = soup.find_all("article", class_="product_pod")

print("Number of books on this page:", len(books))

Number of books on this page: 20


In [8]:
first_book = books[0]

print(first_book.h3.a["title"])

A Light in the Attic


In [9]:
first_book.h3.a["title"]

'A Light in the Attic'

In [10]:
!pip install beautifulsoup4 requests pandas


In [11]:
!pip install beautifulsoup4 requests pandas

In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import os
import re
import statistics

In [13]:
url = "http://books.toscrape.com/"

response = requests.get(url)

print("Status code:", response.status_code)
print(response.text[:300])

Status code: 200
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lan


In [14]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text.strip())

All products | Books to Scrape - Sandbox


In [15]:
books = soup.find_all("article", class_="product_pod")

print("Number of books:", len(books))

Number of books: 20


In [16]:
first_book = books[0]

title = first_book.h3.a["title"]

price = first_book.find("p", class_="price_color").text.strip()

rating = first_book.find("p", class_="star-rating")["class"]

availability = first_book.find(
    "p", class_="instock availability"
).text.strip()

print("Title:", title)
print("Price:", price)
print("Rating:", rating)
print("Availability:", availability)

Title: A Light in the Attic
Price: Â£51.77
Rating: ['star-rating', 'Three']
Availability: In stock


In [17]:
def scrape_page(url, category):
    """
    Scrape all books from one Books to Scrape page.
    """

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    records = []

    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    for book in books:

        # Title
        title_tag = book.h3.a
        title = title_tag.get("title", "").strip()

        # Price
        price_tag = book.find("p", class_="price_color")
        price = price_tag.text.strip() if price_tag else None

        # Rating
        rating_tag = book.find("p", class_="star-rating")

        if rating_tag:
            rating_text = rating_tag.get("class", [])
            rating_word = rating_text[-1] if len(rating_text) > 1 else None
            rating = rating_map.get(rating_word)
        else:
            rating_word = None
            rating = None

        # Availability
        availability_tag = book.find(
            "p",
            class_="instock availability"
        )

        availability = (
            availability_tag.get_text(" ", strip=True)
            if availability_tag
            else None
        )

        records.append({
            "title": title,
            "price": price,
            "star_rating": rating_word,
            "availability": availability,
            "category": category
        })

    return records

In [18]:
all_records = []

base_url = "https://books.toscrape.com/catalogue/page-{}.html"

for page in range(1, 6):

    url = base_url.format(page)

    print("Scraping:", url)

    records = scrape_page(
        url,
        category="All Products"
    )

    all_records.extend(records)

print("\nTotal records scraped:", len(all_records))

Scraping: https://books.toscrape.com/catalogue/page-1.html
Scraping: https://books.toscrape.com/catalogue/page-2.html
Scraping: https://books.toscrape.com/catalogue/page-3.html
Scraping: https://books.toscrape.com/catalogue/page-4.html
Scraping: https://books.toscrape.com/catalogue/page-5.html

Total records scraped: 100


In [19]:
df = pd.DataFrame(all_records)

print(df.shape)
display(df.head())

(100, 5)


,title,price,star_rating,availability,category
0,A Light in the Attic,Â£51.77,Three,In stock,All Products
1,Tipping the Velvet,Â£53.74,One,In stock,All Products
2,Soumission,Â£50.10,One,In stock,All Products
3,Sharp Objects,Â£47.82,Four,In stock,All Products
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,All Products


In [20]:
df["price_gbp"] = (
    df["price"]
    .str.replace("Â£", "", regex=False)
    .astype(float)
)

In [21]:
print(df[["price", "price_gbp"]].head())

     price  price_gbp
0  Â£51.77      51.77
1  Â£53.74      53.74
2  Â£50.10      50.10
3  Â£47.82      47.82
4  Â£54.23      54.23


In [22]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["star_rating"].map(rating_map)

In [23]:
print(df[["star_rating", "rating"]].head())

  star_rating  rating
0       Three       3
1         One       1
2         One       1
3        Four       4
4        Five       5


In [24]:
df["in_stock"] = (
    df["availability"]
    .str.contains("In stock", case=False, na=False)
)

In [25]:
print(df[["availability", "in_stock"]].head())

  availability  in_stock
0     In stock      True
1     In stock      True
2     In stock      True
3     In stock      True
4     In stock      True


In [26]:
GBP_TO_INR = 105.50

df["price_inr"] = df["price_gbp"] * GBP_TO_INR

In [27]:
display(
    df[["title", "price_gbp", "price_inr"]].head()
)

,title,price_gbp,price_inr
0,A Light in the Attic,51.77,5461.735
1,Tipping the Velvet,53.74,5669.570
2,Soumission,50.10,5285.550
3,Sharp Objects,47.82,5045.010
4,Sapiens: A Brief History of Humankind,54.23,5721.265


In [28]:
print(df.isnull().sum())

title           0
price           0
star_rating     0
availability    0
category        0
price_gbp       0
rating          0
in_stock        0
price_inr       0
dtype: int64


In [29]:
print("Missing price:", df["price_gbp"].isnull().sum())
print("Missing rating:", df["rating"].isnull().sum())

Missing price: 0
Missing rating: 0


In [30]:
if df["price_gbp"].isnull().any():
    median_price = df["price_gbp"].median()
    df["price_gbp"] = df["price_gbp"].fillna(median_price)

if df["rating"].isnull().any():
    median_rating = df["rating"].median()
    df["rating"] = df["rating"].fillna(median_rating).round().astype(int)

In [31]:
df = df.dropna(subset=["title", "category"])

In [32]:
df = df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "availability",
        "category"
    ]
]

In [33]:
display(df.head())

,title,price_gbp,price_inr,rating,in_stock,availability,category
0,A Light in the Attic,51.77,5461.735,3,True,In stock,All Products
1,Tipping the Velvet,53.74,5669.570,1,True,In stock,All Products
2,Soumission,50.10,5285.550,1,True,In stock,All Products
3,Sharp Objects,47.82,5045.010,4,True,In stock,All Products
4,Sapiens: A Brief History of Humankind,54.23,5721.265,5,True,In stock,All Products


In [34]:
print(df.dtypes)

title            object
price_gbp       float64
price_inr       float64
rating            int64
in_stock           bool
availability     object
category         object
dtype: object


In [35]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nUnique ratings:")
print(df["rating"].unique())

print("\nRating range:")
print(df["rating"].min(), "to", df["rating"].max())

print("\nCategories:")
print(df["category"].unique())

print("\nMissing values:")
print(df.isnull().sum())

Rows: 100
Columns: 7

Unique ratings:
[3 1 4 5 2]

Rating range:
1 to 5

Categories:
['All Products']

Missing values:
title           0
price_gbp       0
price_inr       0
rating          0
in_stock        0
availability    0
category        0
dtype: int64


In [36]:
os.makedirs("data_pipeline", exist_ok=True)

df.to_csv(
    "data_pipeline/books_cleaned.csv",
    index=False
)

print("Saved cleaned dataset.")

Saved cleaned dataset.


In [37]:
os.listdir("data_pipeline")

['books_cleaned.csv']

In [38]:
db_path = "data_pipeline/books.db"

conn = sqlite3.connect(db_path)

print("Database connected.")

Database connected.


In [39]:
create_categories = """
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
);
"""

conn.execute(create_categories)
conn.commit()

In [40]:
create_books = """
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    availability TEXT,
    category_id INTEGER,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
);
"""

conn.execute(create_books)
conn.commit()

In [41]:
categories = df["category"].dropna().unique()

for category in categories:
    conn.execute(
        """
        INSERT OR IGNORE INTO categories (category_name)
        VALUES (?)
        """,
        (category,)
    )

conn.commit()

In [42]:
pd.read_sql(
    "SELECT * FROM categories",
    conn
)

,category_id,category_name
0,1,All Products


In [43]:
category_lookup = pd.read_sql(
    "SELECT category_id, category_name FROM categories",
    conn
)

display(category_lookup)

,category_id,category_name
0,1,All Products


In [44]:
category_map = dict(
    zip(
        category_lookup["category_name"],
        category_lookup["category_id"]
    )
)

print(category_map)

{'All Products': 1}


In [45]:
df["category_id"] = df["category"].map(category_map)

In [46]:
for _, row in df.iterrows():

    conn.execute(
        """
        INSERT INTO books (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            availability,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """,
        (
            row["title"],
            row["price_gbp"],
            row["price_inr"],
            row["rating"],
            int(row["in_stock"]),
            row["availability"],
            row["category_id"]
        )
    )

conn.commit()

print("Books inserted.")

Books inserted.


In [47]:
books_count = pd.read_sql(
    "SELECT COUNT(*) AS total_books FROM books",
    conn
)

display(books_count)

,total_books
0,100


In [48]:
query1 = """
SELECT title, price_gbp, rating
FROM books
WHERE rating = 5;
"""

result1 = pd.read_sql(query1, conn)

display(result1.head())

,title,price_gbp,rating
0,Sapiens: A Brief History of Humankind,54.23,5
1,Set Me Free,17.46,5
2,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5
3,Rip it Up and Start Again,35.02,5
4,Chase Me (Paris Nights #2),25.27,5


In [49]:
query1 = """
SELECT title, price_gbp, rating
FROM books
WHERE rating = 5;
"""

result1 = pd.read_sql(query1, conn)

display(result1.head())

,title,price_gbp,rating
0,Sapiens: A Brief History of Humankind,54.23,5
1,Set Me Free,17.46,5
2,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5
3,Rip it Up and Start Again,35.02,5
4,Chase Me (Paris Nights #2),25.27,5


In [50]:
query3 = """
SELECT DISTINCT rating
FROM books
ORDER BY rating;
"""

result3 = pd.read_sql(query3, conn)

display(result3)

,rating
0,1
1,2
2,3
3,4
4,5


In [51]:
query4 = """
SELECT title, price_gbp
FROM books
WHERE price_gbp BETWEEN 20 AND 30
ORDER BY price_gbp;
"""

result4 = pd.read_sql(query4, conn)

display(result4.head(10))

,title,price_gbp
0,The Inefficiency Assassin: Time Management Tac...,20.59
1,Shakespeare's Sonnets,20.66
2,In the Country We Love: My Family Divided,22.00
3,America's Cradle of Quarterbacks: Western Penn...,22.50
4,The Boys in the Boat: Nine Americans and Their...,22.60
5,The Requiem Red,22.65
6,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11
7,The Elephant Tree,23.82
8,Olio,23.88
9,The Mindfulness and Acceptance Workbook for An...,23.89


In [52]:
query5 = """
SELECT title, rating
FROM books
WHERE rating IN (4, 5)
ORDER BY rating DESC;
"""

result5 = pd.read_sql(query5, conn)

display(result5.head(10))

,title,rating
0,Sapiens: A Brief History of Humankind,5
1,Set Me Free,5
2,Scott Pilgrim's Precious Little Life (Scott Pi...,5
3,Rip it Up and Start Again,5
4,Chase Me (Paris Nights #2),5
5,Black Dust,5
6,Worlds Elsewhere: Journeys Around Shakespeareâ...,5
7,The Four Agreements: A Practical Guide to Pers...,5
8,The Elephant Tree,5
9,Sophie's World,5


In [53]:
query6 = """
SELECT
    b.title,
    b.price_gbp,
    b.rating,
    c.category_name
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
ORDER BY b.price_gbp DESC
LIMIT 10;
"""

result6 = pd.read_sql(query6, conn)

display(result6)

,title,price_gbp,rating,category_name
0,The Death of Humanity: and the Case for Life,58.11,4,All Products
1,Slow States of Collapse: Poems,57.31,3,All Products
2,Our Band Could Be Your Life: Scenes from the A...,57.25,3,All Products
3,The Past Never Ends,56.50,4,All Products
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1,All Products
5,Masks and Shadows,56.40,2,All Products
6,The Secret of Dreadwillow Carse,56.13,1,All Products
7,The Electric Pencil: Drawings from Inside Stat...,56.06,1,All Products
8,Birdsong: A Story in Pictures,54.64,3,All Products
9,Sapiens: A Brief History of Humankind,54.23,5,All Products


In [54]:
query2 = """
SELECT title, price_gbp, rating
FROM books
ORDER BY price_gbp DESC
LIMIT 10;
"""

queries = {
    "query_1_where": query1,
    "query_2_order_limit": query2,
    "query_3_distinct": query3,
    "query_4_between": query4,
    "query_5_in": query5,
    "query_6_join": query6
}

In [55]:
result2 = pd.read_sql(query2, conn)

display(result2)

,title,price_gbp,rating
0,The Death of Humanity: and the Case for Life,58.11,4
1,Slow States of Collapse: Poems,57.31,3
2,Our Band Could Be Your Life: Scenes from the A...,57.25,3
3,The Past Never Ends,56.50,4
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1
5,Masks and Shadows,56.40,2
6,The Secret of Dreadwillow Carse,56.13,1
7,The Electric Pencil: Drawings from Inside Stat...,56.06,1
8,Birdsong: A Story in Pictures,54.64,3
9,Sapiens: A Brief History of Humankind,54.23,5


In [56]:
os.makedirs("data_pipeline/sql_outputs", exist_ok=True)

In [57]:
with open(
    "data_pipeline/sql_outputs/queries.sql",
    "w"
) as f:

    for name, query in queries.items():

        f.write(f"-- {name}\n")
        f.write(query)
        f.write("\n\n")

In [58]:
result1.to_csv(
    "data_pipeline/sql_outputs/query1_where.csv",
    index=False
)

result2.to_csv(
    "data_pipeline/sql_outputs/query2_order_limit.csv",
    index=False
)

result3.to_csv(
    "data_pipeline/sql_outputs/query3_distinct.csv",
    index=False
)

result4.to_csv(
    "data_pipeline/sql_outputs/query4_between.csv",
    index=False
)

result5.to_csv(
    "data_pipeline/sql_outputs/query5_in.csv",
    index=False
)

result6.to_csv(
    "data_pipeline/sql_outputs/query6_join.csv",
    index=False
)

In [59]:
sql_books = pd.read_sql(
    """
    SELECT title, price_gbp, rating
    FROM books
    WHERE rating >= 4
    """,
    conn
)

display(sql_books.head())

,title,price_gbp,rating
0,Sharp Objects,47.82,4
1,Sapiens: A Brief History of Humankind,54.23,5
2,The Dirty Little Secrets of Getting Your Dream...,33.34,4
3,The Boys in the Boat: Nine Americans and Their...,22.60,4
4,Shakespeare's Sonnets,20.66,4


In [60]:
books_df = pd.read_sql(
    "SELECT * FROM books",
    conn
)

categories_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

In [61]:
merged_df = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

In [62]:
merged_result = merged_df[
    [
        "title",
        "price_gbp",
        "rating",
        "category_name"
    ]
].sort_values(
    "price_gbp",
    ascending=False
).head(10)

In [63]:
display(merged_result)

,title,price_gbp,rating,category_name
68,The Death of Humanity: and the Case for Life,58.11,4,All Products
40,Slow States of Collapse: Poems,57.31,3,All Products
15,Our Band Could Be Your Life: Scenes from the A...,57.25,3,All Products
58,The Past Never Ends,56.50,4,All Products
57,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1,All Products
91,Masks and Shadows,56.40,2,All Products
56,The Secret of Dreadwillow Carse,56.13,1,All Products
67,The Electric Pencil: Drawings from Inside Stat...,56.06,1,All Products
25,Birdsong: A Story in Pictures,54.64,3,All Products
4,Sapiens: A Brief History of Humankind,54.23,5,All Products


In [64]:
result6

,title,price_gbp,rating,category_name
0,The Death of Humanity: and the Case for Life,58.11,4,All Products
1,Slow States of Collapse: Poems,57.31,3,All Products
2,Our Band Could Be Your Life: Scenes from the A...,57.25,3,All Products
3,The Past Never Ends,56.50,4,All Products
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1,All Products
5,Masks and Shadows,56.40,2,All Products
6,The Secret of Dreadwillow Carse,56.13,1,All Products
7,The Electric Pencil: Drawings from Inside Stat...,56.06,1,All Products
8,Birdsong: A Story in Pictures,54.64,3,All Products
9,Sapiens: A Brief History of Humankind,54.23,5,All Products


In [65]:
merged_result

,title,price_gbp,rating,category_name
68,The Death of Humanity: and the Case for Life,58.11,4,All Products
40,Slow States of Collapse: Poems,57.31,3,All Products
15,Our Band Could Be Your Life: Scenes from the A...,57.25,3,All Products
58,The Past Never Ends,56.50,4,All Products
57,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1,All Products
91,Masks and Shadows,56.40,2,All Products
56,The Secret of Dreadwillow Carse,56.13,1,All Products
67,The Electric Pencil: Drawings from Inside Stat...,56.06,1,All Products
25,Birdsong: A Story in Pictures,54.64,3,All Products
4,Sapiens: A Brief History of Humankind,54.23,5,All Products


In [66]:
sql_compare = result6.reset_index(drop=True)

pandas_compare = merged_result.reset_index(drop=True)

print(
    "SQL JOIN and Pandas merge equivalent:",
    sql_compare.equals(pandas_compare)
)

SQL JOIN and Pandas merge equivalent: True


In [67]:
print("========== FINAL VALIDATION ==========")

print("Total books:", len(df))

print(
    "Number of categories:",
    df["category"].nunique()
)

print(
    "Rating range:",
    df["rating"].min(),
    "to",
    df["rating"].max()
)

print(
    "Price data type:",
    df["price_gbp"].dtype
)

print(
    "INR conversion rate:",
    GBP_TO_INR
)

print(
    "Missing values:"
)

print(df.isnull().sum())

print(
    "\nSQL JOIN vs Pandas merge:",
    sql_compare.equals(pandas_compare)
)

========== FINAL VALIDATION ==========
Total books: 100
Number of categories: 1
Rating range: 1 to 5
Price data type: float64
INR conversion rate: 105.5
Missing values:
title           0
price_gbp       0
price_inr       0
rating          0
in_stock        0
availability    0
category        0
category_id     0
dtype: int64

SQL JOIN vs Pandas merge: True


In [68]:
conn.close()

print("Database connection closed.")

Database connection closed.
